Import libraries

In [1]:
import json
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Libraries imported successfully!")

Libraries imported successfully!


Load cleaned documents

In [2]:
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
    / "cleaned_documents.jsonl"
)

documents = []

with open(INPUT_FILE, "r", encoding="utf-8") as file:
    for line in file:
        documents.append(json.loads(line))

print("Total cleaned pages:", len(documents))

Total cleaned pages: 307


Configure chunk size and overlap

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    length_function=len
)

print("Chunk size:", 1000)
print("Chunk overlap:", 150)

Chunk size: 1000
Chunk overlap: 150


Generate chunks

In [4]:
chunked_documents = []

for document in documents:

    chunks = text_splitter.split_text(document["text"])

    for chunk_index, chunk in enumerate(chunks):

        chunked_documents.append({
            "chunk_id": f"{document['source']}_page_{document['page']}_chunk_{chunk_index}",
            "source": document["source"],
            "page": document["page"],
            "chunk_index": chunk_index,
            "text": chunk
        })

print("Total pages:", len(documents))
print("Total chunks:", len(chunked_documents))

Total pages: 307
Total chunks: 1017


Inspect generated chunks

In [5]:
for chunk in chunked_documents[:3]:

    print("\nChunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Text length:", len(chunk["text"]))
    print("-" * 60)
    print(chunk["text"][:500])


Chunk ID: omguide_complete_w-eo-disclaimer.pdf_page_1_chunk_0
Source: omguide_complete_w-eo-disclaimer.pdf
Page: 1
Text length: 141
------------------------------------------------------------
FEDERAL ENERGY MANAGEMENT PROGRAM
Release 3.0
Operations & Maintenance
Best Practices
A Guide to Achieving Operational Efficiency
August 2010

Chunk ID: omguide_complete_w-eo-disclaimer.pdf_page_3_chunk_0
Source: omguide_complete_w-eo-disclaimer.pdf
Page: 3
Text length: 307
------------------------------------------------------------
Release 3.0
Operations & Maintenance
Best Practices
A Guide to Achieving Operational Efficiency
G. P. Sullivan(a)
R. Pugh
A. P. Melendez
W. D. Hunt
August 2010
Prepared by
Pacific Northwest National Laboratory
for the Federal Energy Management Program
U.S. Department of Energy
(a) Efficiency Solutions, LLC

Chunk ID: omguide_complete_w-eo-disclaimer.pdf_page_4_chunk_0
Source: omguide_complete_w-eo-disclaimer.pdf
Page: 4
Text length: 973
-----------------------------

Save chunked documents

In [6]:
OUTPUT_FILE = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
    / "chunked_documents.jsonl"
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as file:

    for chunk in chunked_documents:
        file.write(
            json.dumps(chunk, ensure_ascii=False) + "\n"
        )

print("Chunked documents saved:", OUTPUT_FILE)

Chunked documents saved: d:\FactoryMindAI\knowledge_base\processed\chunked_documents.jsonl


Validate your chunks

In [7]:
chunk_lengths = [
    len(chunk["text"])
    for chunk in chunked_documents
]

print("Total chunks:", len(chunk_lengths))
print("Minimum length:", min(chunk_lengths))
print("Maximum length:", max(chunk_lengths))
print("Average length:", sum(chunk_lengths) / len(chunk_lengths))

Total chunks: 1017
Minimum length: 24
Maximum length: 1000
Average length: 837.5722713864307


In [8]:
assert all(chunk["source"] for chunk in chunked_documents)
assert all(chunk["page"] > 0 for chunk in chunked_documents)
assert all(chunk["text"].strip() for chunk in chunked_documents)

print("Chunk validation successful!")

Chunk validation successful!
